## 7. Git

Every project changes: you add a function, break something, try an idea that does not work, want to go back to what worked yesterday, or work on the same code from two computers or with other people. Copying folders like `project_final`, `project_final_v2` and `project_final_REAL` does not scale. **Git** solves this problem properly. It is the standard tool for tracking changes in code, and **GitHub** is where most of that code lives online.

This section goes from the mental model to the professional workflow:

* **1. Key concepts**: what Git and GitHub are, the **three areas** (working directory, staging area, repository), and what a commit really is.
* **2. Daily workflow (local)**: `init`, `status`, `add`, `commit`, `log`, `diff`, good commit messages and `.gitignore`.
* **3. Undoing things**: `restore`, `--amend`, `revert`, `reset`, plus `reflog` and `stash` as safety nets.
* **4. Branches**: creating, switching, merging and **resolving conflicts**.
* **5. GitHub**: authentication, remotes, `push` / `fetch` / `pull`, Pull Requests, README, Issues and your profile.
* **6. Summary**: a cheatsheet, a daily routine and exercises.

> **How to use this section.** The code cells run real Git commands with IPython's `!` prefix. Everything happens inside a **throwaway "playground" folder** in your temp directory, so you can run the cells in order, break things and re-run the setup cell to start over. Nothing touches this project or your GitHub account. In a real terminal (PowerShell, Git Bash, the VS Code terminal), you type the same commands without the `!`.

### 7.1 Key Concepts

#### **Version Control**

A **version control system (VCS)** records changes to a set of files over time, so you can:

* **Go back** to any previous version of a file or of the whole project.
* See **what** changed, **when**, **who** changed it and **why**.
* Work on **several ideas in parallel** without them interfering with each other (branches).
* **Collaborate**: several people change the same project and merge their work.
* Keep a **backup**: every copy of the repository contains the full history.

Git is a ***distributed*** VCS. Each user has a **complete copy** of the repository, history included, on their own computer. Almost every operation (commit, log, diff, branch) is local and therefore instant, and it works offline. You only need the network to **synchronize** with another copy.

#### **Git vs GitHub**

They are often confused, but they are different things:

| | **Git** | **GitHub** |
|---|---|---|
| What it is | A **program** (command-line tool) | A **web service** (a company, owned by Microsoft) |
| Where it runs | On **your computer** | In the **cloud** (github.com) |
| What it does | Tracks versions: commits, branches, merges, history | **Hosts** Git repositories online and adds collaboration: Pull Requests, Issues, code review, Actions (CI), profiles... |
| Needs the other? | No. Git works perfectly without GitHub | Yes. It is built on top of Git |

Git is the engine and GitHub is one place to park the car and share it.

#### **The Three Areas**

This is **the** key idea. Once it clicks, every command makes sense. A Git project has three areas:

1. **Working directory** (*working tree*): the files you see and edit in your folder. A copy of a version of the project.
2. **Staging area** (*index*): a **draft of the next commit**. File inside `.git/` that contains info. about what will go with your next confirmation. It lets you commit only *some* of your changes.
3. **Repository/git directory**: a project folder that Git tracks, where metadata and your project files with their complete **history** of changes are stored. Defined by a hidden `.git/` folder in the root of the project.

```
  WORKING DIRECTORY            STAGING AREA (index)         REPOSITORY (.git directory)
  Files you edit               Draft of the next commit     History of commits
         │                              │                             │
         │─────────── git add ─────────►│                             │
         │                              │──────── git commit ────────►│
         │                              │                             │
         │◄──── git restore <file> ─────│                             │
         │                              │◄─ git restore --staged ─────│
         │◄──────── git switch <branch> / git restore --source=... ───│
```

A basic Git workflow is:

1. Modify files in your **working directory.**
2. **Add** your files (`git add`) to the **staging** area to prepare them.
3. Confirm changes: take the files as they are in the staging area and stores permanently that snapshot of the project into the **repository.** 

#### **Commits and History**

A **commit** is a **snapshot** ("photo") of the **whole project** at one moment, plus metadata:

* A unique identifier, the **hash** (SHA-1, 40 hexadecimal characters, e.g. `3f9a2c1e...`). Usually the first 7 characters are enough to refer to it.
* **Author** (name + email) and **date**.
* A **message** describing the change.
* A pointer to its **parent** commit(s), the one(s) that came right before.

Because each commit points to its parent, the **history** is a **chain of commits**:

```
  C1 ◄── C2 ◄── C3 ◄── C4      ← main  ← HEAD
```

* **`main`** is a **branch:** just a movable label pointing to the latest commit of a line of work. Every new commit moves it forward. (Older repositories call it `master`.)
* **`HEAD`** means "where you are now": it normally points to the branch you are on.
* `HEAD~1` is the commit before `HEAD`, `HEAD~2` the one before that, and so on.

Some important consequences:

* Commits are **immutable**. "Changing" a commit actually creates a **new** commit with a different hash.
* Git stores snapshots, not "differences", but it is smart about it: unchanged files are not copied again, only referenced.
* Once something is committed, it is **very hard to lose**. Uncommitted work is not protected. **Commit often.**

#### **The Lifecycle of a File**

From Git's point of view, each file in your **working** directory is in one of these states:

```
  untracked ── git add ──► staged ── git commit ──► unmodified (committed)
                           ▲                            │
                           │                        you edit it
                        git add                         ▼
                           └──────────────────────── modified
```

* **Untracked**: Git sees the file but does not track it (a new file, never added).
* **Unmodified**: tracked and identical to the last commit.
* **Modified**: tracked, changed since the last commit, but not staged.
* **Staged**: the change is in the staging area and will go into the next commit.

`git status` tells you exactly which state every file is in. It is the command you will use the most.

#### **Installing and Configuring Git**

* **Install**: on Windows, [Git for Windows].
* **Check**: `git --version`.

Git is mainly used through a **CLI** (Command Line Interface), such as Powershell in Windows. CLI let you execute **all** of Git's commands.

**Initial configuration (once per computer):** every commit records an author, so Git needs your name and email:

```bash
git config --global user.name  "Your Name"
git config --global user.email "you@example.com"     # Use the same email as your GitHub account
git config --global init.defaultBranch main          # New repos start on "main", not "master"
git config --global core.editor "code --wait"        # VS Code as the editor for messages
```

> **Email privacy.** Commits pushed to a public repository show the author's email to everyone. GitHub gives you a private alternative (*Settings → Emails → "Keep my email address private"*), an address like `12345+username@users.noreply.github.com`. Many people use that one in `user.email`.

**Configuration levels.** Settings are read from three places, and the more specific one wins:

| Level | Flag | Applies to | File |
|---|---|---|---|
| System | `--system` | Every user of the computer | Git's install folder |
| Global | `--global` | **You**, in every repository | `~/.gitconfig` |
| Local | `--local` (default) | **One** repository | `.git/config` |

For example, you can have a personal email globally and a work email locally in the work repositories.

**Getting help**: `git help <command>` opens the full manual, and `git <command> -h` prints a short summary of the options.

In [ ]:
# Which Git version is installed, and what is configured?
!git --version
!git config --get user.name
!git config --get init.defaultBranch

# Every setting and the file it comes from (uncomment to see it):
# !git config --list --show-origin

> **Line endings on Windows.** Windows ends lines with `CRLF` (`\r\n`), Linux and macOS with `LF` (`\n`). Git for Windows is usually installed with `core.autocrlf=true`: files are stored as `LF` inside the repository and converted to `CRLF` in your folder. That is why you may see warnings such as *"LF will be replaced by CRLF"*. They are harmless.

### 7.2 Daily Workflow (Local)

#### **Preparing a Playground**

The next cell creates an empty folder `git_playground` in your temp directory and moves the notebook's working directory into it (with IPython's `%cd`). **Re-run it whenever you want to start this section over.** It also defines two small helpers to write and show files from Python. In a real project you would just edit the files in VS Code.

Notebook cells cannot interact with a text editor. If a Git command tries to open one (to write a commit message, for example), the cell would hang. That's why every `commit` here uses `-m`, and the cell sets a couple of environment variables as a safety net.

One more detail: on Windows, `!` lines run in `cmd`, which has no `#` comments. A comment at the end of a `!git ...` line would be passed to Git as extra arguments, so in these cells the comments always go on their own line.

In [ ]:
import os, shutil, stat, tempfile
from pathlib import Path

# Remember where the notebook lives (kept across re-runs) to come back at the end
NOTEBOOK_DIR = globals().get("NOTEBOOK_DIR", Path.cwd())
PLAYGROUND = Path(tempfile.gettempdir()) / "git_playground"

def _force_remove(func, path, exc):           # .git contains read-only files on Windows
    os.chmod(path, stat.S_IWRITE)
    func(path)

os.chdir(NOTEBOOK_DIR)                        # can't delete a folder we are inside of
if PLAYGROUND.exists():
    shutil.rmtree(PLAYGROUND, onexc=_force_remove)
PLAYGROUND.mkdir()

# Safety net: never open an editor from a notebook cell
os.environ["GIT_EDITOR"] = "true"             # "true" = a command that does nothing
os.environ["GIT_MERGE_AUTOEDIT"] = "no"       # merges use the default message

def write(path, text, mode="w"):
    """Create/overwrite (mode='w') or append to (mode='a') a text file."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, mode, encoding="utf-8") as f:
        f.write(text)

def show(path):
    print(Path(path).read_text(encoding="utf-8"))

%cd {PLAYGROUND}

#### **Creating a Repository: `git init`**

`git init` turns a normal folder into a Git repository by creating the hidden **`.git/`** folder inside it. That folder ***is*** the repository: all the history, branches and configuration live there. If you delete `.git/`, your files stay, but the history is gone.

* `git init` creates the repository in the current folder.
* `git init <folder>` creates the folder too. `-b main` names the first branch.

There are two ways to get a Git repository: **create** one (`git init`; take a directory and import it to Git) or **copy** an existing one (`git clone <url>`, covered in the GitHub section).

As a running example, we will build a tiny project that validates **clinical vital signs**.

In [ ]:
!git init -b main vitals-project
%cd vitals-project

# What does Git store in .git/? (You never edit it by hand)
print(sorted(p.name for p in Path(".git").iterdir()))

#### **Checking the State: `git status`**

Each file can be in one of two states: **tracked** and **untracked.** Tracked files are the ones who were in the last snapshot of the project and may be unmodified, modified or staged. Untracked files are the one sin your working directory that were not snapshotted or are staged. 

`git status` answers *"what's going on?"*: which branch you are on, which files are untracked, modified or staged, and it suggests the next commands. Run it constantly, especially before `add` and `commit`.

`git status -s` (*short*) gives a compact view with two columns: the left one is the **staging area** and the right one is the **working directory**.

| Code | Meaning |
|---|---|
| `??` | Untracked |
| `A ` | New file, staged |
| `()M` | Modified, **not** staged |
| `M()` | Modified and staged |
| `MM` | Staged, and then modified again |
| `D ` / `R ` | Deleted / renamed (staged) |

In [ ]:
!git status
print("\n")
!git status -s

In [ ]:
# Let's create the first files of the project
write("vitals.py", '''\
def check_heart_rate(bpm):
    """Classify a resting heart rate (beats per minute)."""
    if bpm < 50:
        return "low"
    if bpm > 100:
        return "high"
    return "normal"
''')

write("README.md", "# Vitals\n\nSmall tools to validate clinical vital signs.\n")

!git status

Both files are **untracked.** Git sees them, but they are **not** part of the project's history yet.

#### **Preparing the Commit: `git add`**

`git add` copies the **current version** of a file into the staging area and starts **tracking** it:

* `git add <file>`: one file (or several).
* `git add <folder>/`: everything inside a folder.
* `git add .`: **everything** that changed in the current folder. It is convenient, but check `git status` first so you do not stage files you did not want.
* `git add -p`: interactive. It asks, chunk by chunk, which changes to stage. It is great for splitting work into clean commits, but it is interactive, so use it in a terminal, not here.

Key detail: `git add` stages the file **as it is at that moment**. If you edit it again afterwards, the new edits are *not* staged until you `git add` again.

In [ ]:
!git add vitals.py
!git status -s

`vitals.py` is staged (`A `), and `README.md` is still untracked (`??`). Let's stage it too.

In [ ]:
!git add .
!git status -s
print("\n")
!git status 

You know those files are being **tracked** and **staged** for commit if you see them after a "Changes to be committed:" header. `git add` can be seen as a "add this content to the next confirmation" command: track new files, stage files.

#### **Saving a Snapshot: `git commit`**

`git commit` takes **what is in the staging area** (not your working directory) and stores it permanently as a new commit. Takes the staging area and **confirms** its changes, kept as modified files in your drive.

* `git commit -m "Message"`: commit with a short message.
* `git commit` (without `-m`) opens your editor to write a longer message (subject + body). In a terminal this is the best way to write good messages.
* `git commit -am "Message"`: stages **all modified tracked files** and commits in one step. It is a handy shortcut, but it does **not include untracked (new)** files.

After the commit, the working directory, the staging area and the last commit are identical, so `git status` reports a clean tree.

In [ ]:
!git commit -m "Add heart rate classification"
print()
!git status

#### **Viewing the History: `git log` and `git show`**

`git log` lists the commits from newest to oldest. Its most useful forms:

| Command | Shows |
|---|---|
| `git log` | Full info: hash, author, date, message |
| `git log --oneline` | One line per commit: short hash + subject |
| `git log --oneline --graph --all` | Draws the branches as a graph (all branches) |
| `git log -n x` | Only the last 'x' commits |
| `git log --stat` | Which files each commit changed, and how much |
| `git log -p <file>` | The full changes (patches) of each commit touching a file |
| `git log --author="Name"` / `--since="2 weeks ago"` / `--grep="word"` | Filters |
| `git show <commit>` | One commit in detail: metadata + its changes (`git show` alone = `HEAD`) |

In [ ]:
!git log

#### **Seeing Changes: `git diff`**

Used to see **exactly** what changed, answering: *What changed and is not staged?* and *What is staged and ready to commit?*.

`git diff` shows **line by line** what changed. Lines starting with `-` were removed and lines with `+` were added. What it compares depends on the arguments, which ties directly to the three areas:

| Command | Compares |
|---|---|
| `git diff` | Working directory vs staging area: **unstaged** changes |
| `git diff --staged` (= `--cached`) | Staging area vs last commit: **staged** changes |
| `git diff HEAD` | Working directory vs last commit: all changes |
| `git diff <commit1> <commit2>` | Two commits (`--stat` for a summary) |
| `git diff main feature` | Two branches |

A good habit is to **review `git diff --staged` before every commit**. It is your last chance to spot accidental changes.

In [ ]:
# Add temperature validation
write("vitals.py", '''
def check_temperature(celsius):
    """Classify a body temperature in degrees Celsius."""
    if celsius < 35.0:
        return "hypothermia"
    if celsius >= 38.0:
        return "fever"
    return "normal"
''', mode="a")

!git diff

In [ ]:
!git add vitals.py
print("--- git diff (unstaged): empty, everything is staged ---")
!git diff
print("--- git diff --staged (what will be committed) ---")
!git diff --staged

In [ ]:
!git commit -m "Add temperature classification"

write("README.md", "\n## Usage\n\n```python\nfrom vitals import check_heart_rate\ncheck_heart_rate(72)  # 'normal'\n```\n", mode="a")
!git commit -am "Document basic usage in README"

!git log --oneline

In [ ]:
# Which files did each commit touch?
!git log --oneline --stat

In [ ]:
# One commit in detail, and the difference between two commits
!git show HEAD~1 --stat
!git diff HEAD~2 HEAD --stat

#### **Moving and Deleting Files: `git mv` and `git rm`**

To delete files from Git, you have to delete them from your **tracked** ones (or better, from the staging area) and then **confirm** the change.

If you rename or delete a file with the file explorer, Git notices, but you then have to stage the change yourself. These commands do both steps at once:

* `git mv old new`: renames the file and stages the rename.
* `git rm <file>`: deletes the file and stages the deletion.

(They also rename/delete the corresponding files in your working directory.)

Another thing you may want is to keep the file on the working directory but delete it from the staging area:
* `git rm --cached <file>`: **stops tracking** the file but **keeps it on disk** (very useful with `.gitignore`, see below).

In [ ]:
write("helpers.py", "def round1(x):\n    return round(x, 1)\n")
!git add helpers.py
!git commit -m "Add helpers module"

!git mv helpers.py utils.py
!git status -s
!git commit -m "Rename helpers to utils"

!git rm utils.py
!git status -s
!git commit -m "Remove unused utils module"

#### **Writing Good Commit Messages**

The history is only useful if you can read it. Months from now, `git log --oneline` should tell a story. Compare these two histories:

```
Bad                               Good
-------------------------------   ---------------------------------------------
a1b2c3d changes                   a1b2c3d Add validation of clinical ranges
b2c3d4e fix                       b2c3d4e Fix fever threshold (38.0 is fever)
c3d4e5f asdf                      c3d4e5f Load patients from CSV instead of Excel
d4e5f6a final version             d4e5f6a Remove unused plotting functions
```

**Rules**:

1. **Short subject line**, with no final period.
2. **Imperative mood**: *"Add"*, *"Fix"*, *"Remove"*, not *"Added"* or *"Adding"*. Git itself writes like this (*"Merge branch..."*, *"Revert..."*). Check it with this test: *"If applied, this commit will **<your subject>**"*.
3. Say **what** changes, specifically: *"Add validation of clinical ranges"*, not *"Update code"*.
4. If more context is needed, leave a **blank line** and write a **body** explaining **why** (the diff already shows *how*). In a terminal, use `git commit` without `-m`, or `-m` twice: `git commit -m "Subject" -m "Body..."`.
5. Make **atomic commits**: one commit = one logical change. *"Fix the fever bug and also reformat everything and add plots"* is three commits. Small commits are easier to understand, review and undo.

Some teams use [**Conventional Commits**](https://www.conventionalcommits.org/), a prefix with the type of change: `feat: add SpO2 validation`, `fix: correct fever threshold`, `docs: ...`, `refactor: ...`, `test: ...`, `chore: ...`. Optional, but can be seen in many repositories.

#### **Ignoring Files: `.gitignore`**

Not everything in your folder should belong in the repository. A **`.gitignore`** file (in the project root, committed like any other file) lists patterns of files that Git must **ignore:** they never show up as untracked, and `git add .` skips them.

**What to ignore,** especially in Python and data projects:

* **Virtual environments**: `venv/`, `.venv/`. They are huge, platform-specific and rebuilt from `requirements.txt`.
* **Generated files**: `__pycache__/`, `*.pyc`, `build/`, `dist/`, `*.egg-info/`.
* **Notebook checkpoints**: `.ipynb_checkpoints/`.
* **Secrets**: `.env` files with passwords, API keys and tokens. **Never commit them.** Commit a `.env.example` with the variable names and fake values instead.
* **Data, above all**: datasets, large CSV files, images, videos, model weights. Git is not designed for heavy files. Every version is kept forever, so the repository grows huge and slow. GitHub warns at 50 MB and rejects files over 100 MB. Download the data separately, and document in the **README** how to get it (a URL, a download script, credentials needed...).
* **Personal files from the OS or editor**: better put in a *global* ignore file, so they do not clutter every project.

**Pattern syntax**:

| Pattern | Ignores |
|---|---|
| `*.log` | Every `.log` file, in any folder |
| `secret.txt` | Any file called `secret.txt`, anywhere |
| `/secret.txt` | Only the one in the root |
| `build/` | Any **folder** called `build` (and its contents) |
| `data/*` | Everything inside the root `data/` |
| `!data/.gitkeep` | Exception: **do** track this file (`!` negates) |
| `**/temp` | `temp` at any depth |
| `# ...` | Comment |

Two details:

* Git does **not** track **empty folders**. To keep a `data/` folder in the repo, add an empty file like `data/.gitkeep` and ignore the rest with `data/*` + `!data/.gitkeep`. (With `data/` alone, the exception would not work: Git does not look inside an ignored folder.)
* You do not have to write `.gitignore` files from scratch. **GitHub** keeps **templates** for every language ([github/gitignore](https://github.com/github/gitignore), such as `Python.gitignore`), and you can pick one when you create a repository.

In [ ]:
# Simulate the typical files of a data project
write("data/patients.csv", "id,bpm,temp\n1,72,36.6\n2,110,38.4\n")
write("data/.gitkeep", "")
write(".env", "API_KEY=super-secret-123\n")
write("__pycache__/vitals.cpython-314.pyc", "binary stuff")
write(".ipynb_checkpoints/analysis-checkpoint.ipynb", "{}")
write("videos/session1.mp4", "pretend this is 2 GB")

!git status -s -uall

Without a `.gitignore`, `git add .` would happily commit the patients' data, the API key and the video. Let's fix that:

In [ ]:
write(".gitignore", """\
# Python
__pycache__/
*.py[cod]

# Virtual environments
venv/
.venv/

# Jupyter
.ipynb_checkpoints/

# Secrets (commit a .env.example instead)
.env

# Data and media: download them separately (see README)
data/*
!data/.gitkeep
videos/
*.mp4
""")

!git status -s -uall

Only `.gitignore` and `data/.gitkeep` remain. We can check *why* a file is ignored (which line of which file matches it) with `git check-ignore -v`, and list the ignored files with `git status --ignored`:

In [ ]:
!git check-ignore -v data/patients.csv .env videos/session1.mp4
!git status -s --ignored

In [ ]:
!git add .
!git commit -m "Add .gitignore for Python, secrets and data"

**Common pitfall: `.gitignore` does not affect files that are already tracked.** If you committed a file and *then* added it to `.gitignore`, Git keeps tracking it. You have to untrack it with `git rm --cached` (the file stays on your disk):

In [ ]:
write("settings.local.json", '{"db_password": "1234"}')
!git add settings.local.json
# Oops
!git commit -m "Add local settings"

write(".gitignore", "\n# Local configuration\nsettings.local.json\n", mode="a")
write("settings.local.json", '{"db_password": "5678"}')
# Still tracked: the change shows up anyway
!git status -s

!git rm --cached settings.local.json
!git add .gitignore
!git commit -m "Stop tracking local settings"
!git status -s
print("Still on disk:", Path("settings.local.json").exists())

> **⚠️ If you push a secret, consider it compromised.** `git rm --cached` removes the file from *future* commits, but the old commit still contains the password, and anyone can read the history. If it was pushed to GitHub (even for a minute; bots scan public repositories for keys), **revoke/rotate the key immediately**. Removing it from the whole history is possible ([`git filter-repo`](https://github.com/newren/git-filter-repo), BFG), but rotating the secret is what actually protects you. GitHub's *push protection* blocks some known key formats, but don't rely on it.

> **Notebooks and Git.** A `.ipynb` is a JSON file that stores **outputs** (including images, and possibly data you printed) and execution counts. Diffs become noisy and the file can leak data. Common practices:
> * Clear the outputs before committing, or install [`nbstripout`](https://github.com/kynan/nbstripout) (`nbstripout --install` does it automatically on each commit).
> * Use [`nbdime`](https://nbdime.readthedocs.io/) for readable notebook diffs and merges (VS Code also shows notebook diffs cell by cell).
> * Move reusable code into `.py` modules and keep notebooks for exploration and presentation.

> **Large files, if you really need them in version control.** [Git LFS](https://git-lfs.com/) keeps large files outside the normal history (GitHub supports it, with storage limits). For datasets and ML models, [DVC](https://dvc.org/) versions data stored in S3, Google Drive, etc., next to your Git repository.

### 7.3 Undoing Things

#### **Which Command Do I Need?**

Almost everything can be undone in Git. The right command depends on **where the change is**:

| Situation | Command | Danger |
|---|---|---|
| I edited a file and want to **discard** the changes | `git restore <file>` | ⚠️ The uncommitted edits are lost |
| I staged a file by mistake **(unstage** it) | `git restore --staged <file>` | Safe, the edits stay in the file |
| I want to **fix the last commit** (message, or a forgotten file) and **have not pushed** it | `git commit --amend` | Rewrites history (only for local commits) |
| I want to **undo a commit** that is already shared/pushed | `git revert <commit>` | **Safe:** adds a new commit |
| I want to **throw away local commits**, or move the branch back | `git reset [--soft / --mixed / --hard] <commit>` | `--hard` deletes uncommitted work |
| I want to **save my changes for later** and get a clean tree | `git stash` | Safe |
| "I broke everything, I want my commit back" | `git reflog` | It's your safety net |

<br>

> **`git checkout`:** older tutorials use `git checkout` for everything (switching branches, discarding changes...). Since Git 2.23 its two jobs are split into clearer commands: **`git switch`** (branches) and **`git restore`** (files). `checkout` still works, but prefer the new ones.

#### **Discarding Changes in a File: `git restore`**

`git restore <file>` overwrites the file in your working directory with its version from the staging area (which is the last commit, if nothing is staged). **The discarded edits are not saved anywhere,** because they were never committed. Use it when you have made a mess and want to go back to the last good version.

* `git restore .` discards **all** unstaged changes.
* `git restore --source=HEAD~2 <file>` brings back the version of a file from an older commit.

In [ ]:
write("vitals.py", "\n# I'm going to rewrite everything... bad idea\nprint('debug')\n", mode="a")
!git status -s

!git restore vitals.py
!git status -s
print("Last lines:", Path("vitals.py").read_text().splitlines()[-2:])

#### **Unstaging a Staged File: `git restore --staged`**

`git restore --staged <file>` takes the file **out of the staging area**, so it will not go into the next commit. **The edits stay in your file.** Basically the inverse of `git add`.

In [ ]:
write("vitals.py", "\n# TODO: add blood pressure\n", mode="a")
!git add vitals.py
# "M ": staged
!git status -s

!git restore --staged vitals.py
# " M": modified but no longer staged; the edit is still there
!git status -s

# (and now we discard it too)
!git restore vitals.py

#### **Fixing the Last Commit: `git commit --amend`**

Typo in the message? Forgot to add a file? `git commit --amend` **replaces the last commit** with a new one that contains the current staging area (plus anything you just added) and a new message (`-m`), or the same message (`--no-edit`).

The amended commit gets a **new hash:** it is a different commit. That is no problem while the commit only exists on your computer. **Do not** amend commits you have already **pushed,** because other copies of the repository still have the old one (see the golden rule below).

In [ ]:
write("vitals.py", '''
def check_respiratory_rate(breaths_per_min):
    """Classify a respiratory rate (breaths per minute)."""
    if breaths_per_min < 12:
        return "low"
    if breaths_per_min > 20:
        return "high"
    return "normal"
''', mode="a")
# Typo, and we forgot the README
!git commit -am "Add respiratry rate clasification"
!git log --oneline -1

write("README.md", "\nAlso classifies respiratory rate.\n", mode="a")
!git add README.md
!git commit --amend -m "Add respiratory rate classification"
# New hash, both files
!git log --oneline -1 --stat

#### **Undoing a Commit Safely: `git revert`**

`git revert <commit>` **does not delete anything.** It creates a **new commit** that applies the exact opposite of the given commit: added lines are removed and removed lines are added back. The history keeps both the mistake and its fix:

```
  C1 ── C2 ── C3(bad) ── C4 = "Revert C3"      ← main
```

That is why it is the **safe way** to undo something that is **already pushed / shared:** nobody's history is rewritten. It can revert any commit, not only the last one. In a terminal it opens the editor for the message, and `--no-edit` accepts the default one.

In [ ]:
# A wrong change gets committed
text = Path("vitals.py").read_text().replace("if bpm > 100:", "if bpm > 300:")
write("vitals.py", text)
!git commit -am "Raise heart rate upper limit"
print()

# Undo it with a new commit
!git revert HEAD --no-edit
print()
!git log --oneline -3
print()
!git show HEAD

#### **Moving the Branch Back: `git reset`**

`git reset <commit>` **moves the current branch** (and `HEAD`) to another commit. The commits after it disappear from the branch. The **mode** decides what happens to the changes those commits contained, again in terms of the three areas:

| Mode | Branch moves | Staging area | Working directory | Use case |
|---|---|---|---|---|
| `--soft` | ✔ | Untouched: the changes stay **staged** | Untouched | Redo the last commit(s), squash 3 commits into 1 |
| `--mixed` (default) | ✔ | Reset: the changes become **unstaged** | Untouched | "Uncommit" and re-organize what goes into each commit |
| `--hard` | ✔ | Reset | **Reset: the changes are deleted** ⚠️ | Throw everything away and go back to that commit |

```
Before:   C1 ── C2 ── C3 ── C4   ← main ← HEAD
git reset --soft HEAD~2
After:    C1 ── C2               ← main ← HEAD     (changes of C3 and C4: staged)
```

**Use it with care:**

* `reset` **rewrites history**. Only use it on commits you **have not pushed**. For shared commits, use `revert`.
* `--hard` also **deletes uncommitted changes** in tracked files, and those **cannot** be recovered: they were never in a commit. Run `git status` before a `--hard`.
* (Untracked files are never touched by `reset`. To delete those, use `git clean -n` to preview and then `git clean -fd`. Be careful, that is also **permanent.)**

In [ ]:
# Two experimental commits
write("experiment_a.py", "print('A')\n")
!git add experiment_a.py
!git commit -m "Try experiment A"
write("experiment_b.py", "print('B')\n")
!git add experiment_b.py
!git commit -m "Try experiment B"
!git log --oneline -3

In [ ]:
# --soft: the last commit is undone, but its changes are still staged
!git reset --soft HEAD~1
print()
!git log --oneline -2
print()
!git status -s

In [ ]:
# --mixed (default): undo one more; now nothing is staged (new files become untracked again)
!git reset HEAD~1
!git log --oneline -2
!git status -s

In [ ]:
# Let's commit both as a single commit... and then delete it completely with --hard
!git add experiment_a.py experiment_b.py
!git commit -m "Try experiments A and B"

!git reset --hard HEAD~1
!git log --oneline -2
!git status -s
print("experiment_a.py exists?", Path("experiment_a.py").exists())

#### **Your Safety Net: `git reflog`**

The "deleted" commit is not really gone. Git keeps a log of **every position `HEAD` has been at** (commits, resets, switches, merges...) for about 90 days. `git reflog` lists it, and you can jump back to any entry. `HEAD@{1}` means "where HEAD was one move ago".

So **anything that was committed can almost always be recovered**. Things that were never committed (and then deleted with `restore` or `reset --hard`) cannot. That's one more reason to commit often.

*(In the cell below, the braces are doubled, `HEAD@{{1}}`, only because IPython's `!` treats `{...}` as a Python expression. In a terminal, write `HEAD@{1}`.)*

In [ ]:
!git reflog -6

In [ ]:
# Go back to the commit we "lost"
!git reset --hard HEAD@{{1}}
!git log --oneline -2
print("experiment_a.py exists?", Path("experiment_a.py").exists())

# We didn't really want it, so drop it again
!git reset --hard HEAD~1

#### **Saving Work for Later: `git stash`**

You are halfway through something and need a clean working directory *right now*, to switch branch, pull, or fix an urgent bug. You do not want to commit half-done work. `git stash` **saves your uncommitted changes** on a stack and cleans the working directory. Later, you can bring them back:

* `git stash` (or `git stash push -m "description"`): save and clean.
* `git stash list`: see what is saved.
* `git stash pop`: re-apply the last stash and remove it from the stack (`apply` keeps it).
* `git stash -u`: include untracked files too.

In [ ]:
write("vitals.py", "\n# Work in progress: oxygen saturation...\n", mode="a")
!git status -s
print()

!git stash push -m "WIP oxygen saturation"
# Clean
!git status -s
print()
!git stash list

# Changes back
!git stash pop
print()
!git status -s
print()
# (discard them, to continue cleanly)
!git restore vitals.py

> **Never rewrite history that has already been shared** (pushed). `--amend` and `reset` create *new* commits that replace the old ones. If someone else (or your other computer) already has the old ones, the histories diverge, and fixing that is a painful mess. Rewrite your **local** commits as much as you like, and use **`revert`** for published ones.

### 7.4 Branches

#### **Branches**

A **branch** is an **independent line of work**. Under the hood it is surprisingly simple: a **movable label that points to a commit.** Creating a branch is instant and costs almost nothing.

```
                     C4 ── C5      ← other branch
                    /
  C1 ── C2 ── C3 ──┘               ← main  ← HEAD
```

* Each new commit moves forward **the branch you are on** (the one `HEAD` points to).
* The `main` branch is the default one created by `git init` (was called `master` before).
* **Switching branch** changes the files in your working directory to that branch's snapshot.

#### **Creating and Switching Branches: `git branch` and `git switch`**

| Command | Does |
|---|---|
| `git branch` | List local branches (`*` marks the current one). `-a` includes remote ones, `-v` the last commit |
| `git branch <name>` | Create a branch (but stay where you are) |
| `git switch <name>` | Switch to a branch |
| `git switch -c <name>` | **Create and switch** in one step (the most common) |
| `git switch -` | Go back to the previous branch |
| `git branch -d <name>` | Delete a branch that is already merged (`-D` forces it, even if not merged) |
| `git branch -m <new>` | Rename the current branch |

Before switching, commit or stash your changes. Git refuses to switch if your uncommitted edits would be overwritten.

**Branch names** should be short, descriptive, in lowercase with hyphens (-), often with a prefix: `feature/spo2-validation`, `fix/fever-threshold`, `docs/install-guide`, `experiment/random-forest`.

In [ ]:
!git branch
!git switch -c feature/oxygen-saturation

write("vitals.py", '''
def check_oxygen_saturation(spo2):
    """Classify peripheral oxygen saturation (%)."""
    if spo2 < 90:
        return "critical"
    if spo2 < 95:
        return "low"
    return "normal"
''', mode="a")
!git commit -am "Add oxygen saturation classification"

!git branch -v
!git log --oneline --graph --all -4

In [ ]:
# Back on main, the new function doesn't exist: the working directory changed
!git switch main
print("check_oxygen_saturation" in Path("vitals.py").read_text())

#### **Merging: `git merge`**

When the work in a branch is ready, you can **merge** it into another branch: first switch to the branch that **receives** the changes, then run `git merge <other-branch>`. There are two cases:

**1. Fast-forward:** if `main` has **not** moved since the branch was created, Git simply **moves the `main` label forward.** No new commit is needed.

```
Before:  C1 ── C2 ── C3          ← main        After:  C1 ── C2 ── C3 ── C4   ← main, feature
                       └── C4    ← feature
```

**2. Three-way merge (merge commit):** if **both** branches have new commits, Git combines them by creating a **merge commit,** which has **two parents.**

```
Before:  C1 ── C2 ── C3 ── C5        ← main     After:  C1 ── C2 ── C3 ── C5 ── M   ← main
                       └── C4        ← feature                      └── C4 ──┘
```

If the branches changed **different parts** of the files, Git combines them automatically. If they changed **the same lines,** you get a **conflict** (next subsection).

In [ ]:
# Case 1: fast-forward
!git merge feature/oxygen-saturation
# Merged, the label is no longer needed
!git branch -d feature/oxygen-saturation
print()
!git log --oneline --graph -3

In [ ]:
# Case 2: both branches advance, so a merge commit is needed
!git switch -c docs/install-guide
write("README.md", "\n## Installation\n\n```bash\ngit clone <url>\ncd vitals-project\n```\n", mode="a")
!git commit -am "Add installation instructions"
print()

!git switch main
write("vitals.py", '\n\nVERSION = "0.1.0"\n', mode="a")
!git commit -am "Add version number"
print()

# --no-edit: accept the default merge message
!git merge docs/install-guide --no-edit
!git branch -d docs/install-guide
!git log --oneline --graph -5

#### **Merge Conflicts**

A **conflict** happens when two branches changed **the same lines** of the same file (or one branch edited a file that the other deleted). Git cannot know which version is right, so it **stops the merge and asks you**. Conflicts can scare, but they are just a question: *"Which of these do you want?"* Nothing is lost.

When it happens:

1. `git merge` says `CONFLICT (content): Merge conflict in <file>`, and `git status` lists the file under *"Unmerged paths"* (`UU` in `-s`).
2. Git writes **both versions** into the file, between **conflict markers:**
   ```
   <<<<<<< HEAD
   your version (the branch you are on, e.g. main)
   =======
   the incoming version (the branch being merged)
   >>>>>>> feature/x
   ```
3. **You edit the file** and leave it as it should be: one version, the other, or a mix of both. **Delete all the markers.**
4. `git add <file>` marks the conflict as resolved.
5. `git commit` finishes the merge (with the default message).

Panic button: **`git merge --abort`** cancels the merge and puts everything back as it was before.

> **VS Code** highlights conflicts with buttons (*Accept Current / Accept Incoming / Accept Both*) and has a 3-way *merge editor*. It is much more comfortable than editing markers by hand. The process underneath is the same.

The best way to learn how to handle them is to **cause one on purpose:**

In [ ]:
# Shared starting point on main: a file with the clinical limits
write("limits.py", """\
HEART_RATE_MIN = 50
HEART_RATE_MAX = 100
FEVER_THRESHOLD = 38.0
""")
!git add limits.py
!git commit -m "Extract clinical limits into limits.py"

# Branch: someone wants stricter limits
!git switch -c feature/strict-limits
write("limits.py", Path("limits.py").read_text().replace("HEART_RATE_MAX = 100", "HEART_RATE_MAX = 90"))
!git commit -am "Lower maximum heart rate to 90"

# Meanwhile on main: someone changes THE SAME LINE differently
!git switch main
write("limits.py", Path("limits.py").read_text().replace("HEART_RATE_MAX = 100", "HEART_RATE_MAX = 110"))
!git commit -am "Raise maximum heart rate to 110 for athletes"

!git log --oneline --graph --all -4

In [ ]:
!git merge feature/strict-limits

In [ ]:
!git status
show("limits.py")

Git merged everything it could. Only the `HEART_RATE_MAX` line is in conflict: `HEAD` (main) says 110, and the branch says 90. Let's say to agree on 100. We write the resolved version **without markers,** then `add` and `commit`:

In [ ]:
write("limits.py", """\
HEART_RATE_MIN = 50
HEART_RATE_MAX = 100          # Agreed default; adjust per patient group if needed
FEVER_THRESHOLD = 38.0
""")
!git add limits.py
!git status -s
# Finishes the merge with the default message
!git commit --no-edit
!git branch -d feature/strict-limits
!git log --oneline --graph -5

Some tips to have fewer (and smaller) conflicts:

* Keep branches **short-lived:** merge often, and do not let a branch live for long.
* **Update your branch** with the latest `main` regularly (`git merge main` while on your branch), so conflicts show up early and small.
* Make small, focused commits, and avoid "reformat the whole file" commits mixed with real changes.

#### **A Simple Branching Workflow**

For personal projects and small teams, this flow (close to *GitHub Flow*) is all you need:

1. **`main` is always stable:** it works, the tests pass, and you could show it to anyone.
2. **One branch per feature, fix or experiment:** `git switch -c feature/xyz` from an up-to-date `main`.
3. Commit on the branch in small steps.
4. When it is done, **merge it into `main`**, ideally through a **Pull Request** on GitHub (next section), even if you work alone.
5. Delete the branch. Experiments that did not work are deleted without merging (`git branch -D`), and `main` was never touched.

```
main      ●───────●───────────────●──────────●───── (always works)
           \             /           \       /
feature/a   ●──●──●──●──┘             \     /
                             fix/b     ●──●┘
```

> **`git rebase`** (for later). Instead of a merge commit, `git rebase main` "replays" your branch's commits on top of the latest `main`, giving a linear history. It is powerful and common in teams, but it **rewrites** the branch's commits. Master `merge` first. [Pro Git, ch. 3.6](https://git-scm.com/book/en/v2/Git-Branching-Rebasing) explains rebase very well.

> **Detached HEAD.** If you switch to a commit instead of a branch (`git switch --detach a1b2c3d`, or `git checkout a1b2c3d`) to look at an old version, Git warns you that you are in *"detached HEAD"* state. It ids fine for looking around. If you make commits there, create a branch (`git switch -c name`) before leaving, or they will be hard to find (only `reflog` will remember them).

### 7.5 GitHub

#### **Remote Repositories**

A **remote** is **another copy of the repository** (a version of your project) hosted somewhere else, usually on GitHub (i.e. on the Internet). Your local repository and the remote are **independent.** Nothing syncs automatically. You decide when to exchange commits:

```
        YOUR COMPUTER                                         GITHUB
  ┌─────────────────────────────┐                     ┌─────────────────────┐
  │ working dir → staging → repo│ ──── git push ───►  │   remote repository │
  │                  main       │                     │       main          │
  │                  origin/main│ ◄─── git fetch ──── │                     │
  │                             │ ◄─── git pull  ──── │  (= fetch + merge)  │
  └─────────────────────────────┘                     └─────────────────────┘
```

* **`origin`** is the conventional **name** of the main remote: an alias for its URL, so you do not type it every time.
* **`origin/main`** is a ***remote-tracking branch*:** your **local, read-only memory** of where `main` was on GitHub the last time you talked to it. `fetch` updates it.
* **`git push`** uploads your new commits to the remote.
* **`git fetch`** downloads the remote's new commits into `origin/...` **without touching your branches or files.** It is always safe.
* **`git pull`** = `git fetch` + `git merge origin/main` into your current branch. It downloads *and* integrates.

#### **Creating an Account and a Repository**

1. Create an account on [github.com](https://github.com). Choose a **professional username:** it is in the URL of every project (`github.com/username/project`).
2. **New repository** (`+` button):
   * **Name**: short, lowercase, with hyphens: `vitals-validator`, `ecg-classifier`.
   * **Description:** one line saying what it does.
   * **Public** (visible to anyone, good for a portfolio) or **Private**.
   * Initialize with README / `.gitignore` / license? **If you already have a local project, leave the repository empty.** Otherwise the first push conflicts with GitHub's initial commit. If you are starting from zero, initialize it and then `clone` it.
3. GitHub then shows the URL in two forms:
   * HTTPS: `https://github.com/username/vitals-validator.git`
   * SSH: `git@github.com:username/vitals-validator.git`

#### **Authentication**

GitHub needs to know it is **you** before accepting a `push`. You have three options:

**Option A: GitHub CLI (`gh`), the easiest.** GitHub's official command-line tool. It logs you in through the browser and configures Git for you. It is also useful later for repos, PRs and issues from the terminal.
```bash
winget install --id GitHub.cli        # Windows
gh auth login                         # GitHub.com → HTTPS → "Login with a web browser"
gh auth status                        # Check
```

**Option B: SSH key.** A key pair: the **private** key stays on your computer (never share it), and the **public** key is uploaded to GitHub. Once it is set up, you never type anything again.
```bash
ssh-keygen -t ed25519 -C "you@example.com"     # Enter to accept the default path; a passphrase is recommended
cat ~/.ssh/id_ed25519.pub                       # Copy this PUBLIC key (the .pub file)
# GitHub → Settings → SSH and GPG keys → New SSH key → paste it
ssh -T git@github.com                           # Test: "Hi username! You've successfully authenticated..."
```
Then use the **SSH URLs** (`git@github.com:...`).

**Option C: HTTPS + Git Credential Manager.** Git for Windows includes it. The first time you push to an HTTPS URL, a browser window asks you to log in, and the credentials are stored securely afterwards. (The manual alternative is a *Personal Access Token* used as a password, but A or C are simpler.)

**Two-factor authentication (2FA)**: GitHub requires it for accounts that contribute code. Turn it on in *Settings → Password and authentication* and keep the recovery codes somewhere safe.

#### **Connecting Local and Remote: `git remote` and `git clone`**

There are two starting points:

**1. You already have a local project** (`git init`, some commits) and want to upload it to a new, **empty** GitHub repository:
```bash
git remote add origin git@github.com:username/vitals-validator.git   # Register the remote with the name "origin"
git remote -v                                                        # Check (fetch and push URLs)
git push -u origin main                                              # First push
```
`-u` (`--set-upstream`) links your local `main` to `origin/main`. From then on, plain **`git push`** and **`git pull`** know where to go.

**2. The repository already exists on GitHub** (yours, or someone else's) and you want it on your computer:
```bash
git clone git@github.com:username/vitals-validator.git       # Creates the folder vitals-validator/
git clone https://... my-folder                                    # ...or with another name
```
`clone` downloads **the whole history,** configures `origin` automatically, and checks out the default branch. No `init` or `remote add` is needed.

Other useful commands: `git remote rename`, `git remote remove origin`, and `git remote set-url origin <new-url>` (e.g. to change from HTTPS to SSH).

#### **Simulating GitHub Locally**

To practice `push` / `fetch` / `pull` without an account or a network, we can use a ***bare* repository** as the "GitHub". A bare repository is a repository with **only the `.git` contents and no working directory**, which is exactly what GitHub stores on its servers. Git accepts a folder path as a remote URL, and the commands are **identical** to using a real GitHub URL.

We will pretend to have:

* `remote.git`: "GitHub".
* `vitals-project`: our repository, on the "desktop computer".
* `laptop`: a clone of the same project on another computer (it could also be a colleague).

In [ ]:
%cd {PLAYGROUND}
!git init --bare -b main remote.git

%cd {PLAYGROUND / "vitals-project"}
!git remote add origin ../remote.git
!git remote -v

#### **Uploading: `git push`**

`git push` sends your **new commits** of the current branch to the remote and moves the branch there. Only commits travel. Uncommitted changes stay on your computer.

* `git push -u origin main`: the first time (sets the upstream).
* `git push`: afterwards.
* `git push -u origin feature/xyz`: upload a branch, typically to open a Pull Request.

In [ ]:
!git push -u origin main
# [origin/main]: main is linked to origin/main
!git branch -vv
# origin/main points to the same commit as main
!git log --oneline -3 --all

#### **Downloading: `git clone`, `git fetch` and `git pull`**

Now the "laptop" clones the project, makes a commit and pushes it. Back on the desktop, our local repository does not know about it yet.

In [ ]:
%cd {PLAYGROUND}
!git clone remote.git laptop
%cd {PLAYGROUND / "laptop"}
# Local config: only for this repository
!git config user.name "Laptop"
!git log --oneline -3

write("vitals.py", '''
def check_blood_pressure(systolic, diastolic):
    """Classify blood pressure (mmHg)."""
    if systolic >= 140 or diastolic >= 90:
        return "hypertension"
    if systolic < 90 or diastolic < 60:
        return "hypotension"
    return "normal"
''', mode="a")
!git commit -am "Add blood pressure classification"
!git push

In [ ]:
# Back on the desktop: fetch downloads the new commit, but our files and main do not change
%cd {PLAYGROUND / "vitals-project"}
!git fetch
print()

# "Your branch is behind 'origin/main' by 1 commit"
!git status
print()
!git log --oneline --graph --all -3

In [ ]:
# Review what is coming before integrating it...
!git diff main origin/main --stat
print()

# ...and integrate it. Here pull = merge of origin/main (a fast-forward)
!git pull
print()
!git log --oneline -2

**`fetch` or `pull`?** `git fetch` is always safe: it only updates your view of the remote, so you can review (`git log main..origin/main`, `git diff`) before merging. `git pull` does both steps at once and is the everyday choice. 

Make it a habit to `git pull` before starting to work and before creating a branch, so you always start from the latest version.

> Since Git 2.27, if your branch and the remote have both diverged, `git pull` asks you to choose how to integrate (merge or rebase). Configure it once with `git config --global pull.rebase false` (always merge, the classic behaviour) or `true` (rebase).

#### **When the Push Is Rejected**

A very common situation: you try to push, but someone (or your other computer) pushed first. The remote has commits you do not have, so Git **rejects** the push to avoid losing them:

```
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs ...
hint: Updates were rejected because the remote contains work that you do not have locally.
```

The fix: **`git pull`** (integrate their work, resolving conflicts if there are any), then **`git push`** again. **Do not use `git push --force`** to "fix" this, because it overwrites the remote and deletes the other person's commits.

In [ ]:
# The laptop pushes a change...
%cd {PLAYGROUND / "laptop"}
!git pull
write("README.md", "\n## Authors\n\nThe vitals team.\n", mode="a")
!git commit -am "Add authors section to README"
!git push

# ...while the desktop makes a different commit without pulling first
%cd {PLAYGROUND / "vitals-project"}
write("limits.py", "SPO2_LOW = 95\nSPO2_CRITICAL = 90\n", mode="a")
!git commit -am "Add oxygen saturation limits"
# Rejected
!git push

In [ ]:
# Integrate with a merge (set once; usually --global)
!git config pull.rebase false
!git pull 
# Now it works
!git push
!git log --oneline --graph -5

#### **Pull Requests**

A **Pull Request (PR)** is GitHub's way of saying *"I have a branch with changes, and I propose merging it into `main`"*. It is not a Git command. It is a GitHub page built around a branch, with:

* The **description** of the change (what and why).
* The full **diff**, with line-by-line **review comments**.
* **Automatic checks** (tests, linters with GitHub Actions) that run on every push to the branch.
* A **Merge** button that performs the merge on GitHub.

**Use them even if you work alone.** It is how virtually every company and open-source project works. It forces you to review your own diff before it reaches `main`, keeps `main` clean, and leaves a documented record of every change. It also looks good on your profile.

**The complete flow:**
```bash
git switch main
git pull                                   # 1. Start from the latest main
git switch -c feature/spo2-alerts          # 2. A branch for the change

# ...edit, git add, git commit (several times)...

git push -u origin feature/spo2-alerts     # 3. Upload the branch

# 4. On GitHub: "Compare & pull request" → Title + Description → Create pull request
#    (or from the terminal: gh pr create --fill)
# 5. Review the diff, fix things with more commits (git push again: the PR updates itself)
# 6. Merge pull request on GitHub (or: gh pr merge --squash --delete-branch)

git switch main
git pull                                   # 7. Bring the merged main to your computer
git branch -d feature/spo2-alerts          # 8. Clean up the local branch
```

GitHub offers **three ways to merge** a PR: *Create a merge commit* (keeps every commit plus a merge commit), ***Squash and merge*** (combines all the PR's commits into a single commit on `main`, which keeps the history clean and is very popular), and *Rebase and merge* (replays the commits linearly).

**Forks:** to contribute to a repository where you **cannot** push (someone else's open-source project), you first **fork** it (your own copy on GitHub), push your branch to your fork, and open the PR from there to the original.

Let's run the Git part of steps 1–3 against our "GitHub":

In [ ]:
!git switch -c feature/spo2-alerts
write("vitals.py", '''
def needs_oxygen_alert(spo2):
    """True if the saturation requires immediate attention."""
    return check_oxygen_saturation(spo2) == "critical"
''', mode="a")
!git commit -am "Add alert for critical oxygen saturation"
# On GitHub, now you would open the PR
!git push -u origin feature/spo2-alerts
# Local branches + remote-tracking ones (remotes/origin/...)
!git branch -a

#### **README.md**

The **README** is the **first thing anyone sees** when they open your repository (GitHub renders it below the file list). A repository without a README looks abandoned, however good the code is. It is written in **Markdown** and should answer these questions quickly:

1. **What** does it do, and **why** does it exist? (one or two sentences, maybe a screenshot or a plot)
2. **How do I install it?** (requirements, `pip install -r requirements.txt`...)
3. **How do I use it?** (a minimal example that can be copied and pasted)
4. For data projects: **where does the data come from** and how to get it, since it isn't in the repository.
5. Project structure, results, license, author/contact.

A template:

````markdown
# Vitals Validator

Classifies clinical vital signs (heart rate, temperature, SpO₂, blood pressure)
as normal or abnormal, following standard adult reference ranges.

## Installation

```bash
git clone https://github.com/username/vitals-validator.git
cd vitals-validator
python -m venv .venv
.venv\Scripts\activate          # Windows  (Linux/macOS: source .venv/bin/activate)
pip install -r requirements.txt
```

## Usage

```python
from vitals import check_heart_rate
check_heart_rate(72)   # 'normal'
```

## Data

The dataset is not included in the repository (size and privacy).
Download it from <URL> and place it in `data/patients.csv`.

## Project structure

```
vitals/        source code
notebooks/     exploration
tests/         tests
```

## License

MIT
````

Remember to keep the README up to date when the project changes.

#### **Issues**

**Issues** are GitHub's to-do list and bug tracker for a repository. Each issue has a number (`#x`), a title, a description, comments, **labels** (`bug`, `enhancement`, `documentation`, `good first issue`...), assignees and milestones.

* Use them as your **personal backlog,** even alone: *"Add SpO₂ validation"*, *"Fever threshold is wrong for children"*.
* A good bug report includes **what you expected**, **what happened**, and **how to reproduce it** (steps, input, error traceback).
* **Link them to your work**: writing `#12` in a commit message or a PR creates a link. Writing **`Closes #12`** (or `Fixes #12`, `Resolves #12`) in a **PR description** closes the issue automatically when the PR is merged into `main`.
* From the terminal: `gh issue create`, `gh issue list`, `gh issue view 12`.

Typical cycle: **Issue #12** → branch `fix/12-fever-threshold` → commits → **PR** "Fix fever threshold (Closes #12)" → merge → the issue closes itself.

#### **A Clean GitHub Profile**

Recruiters and collaborators *will* look at your GitHub, often before your CV. Keep it tidy:

* **Profile:** a real photo or a neutral avatar, your name, a one-line bio, location, and links (LinkedIn, website).
* **Profile README:** create a public repository with **exactly your username** as its name (`username/username`). Its `README.md` appears at the top of your profile. Use it for a short introduction: who you are, what you work on, your stack, and highlighted projects.
* **Pinned repositories:** pin your **6 best projects**. Each one should have a description, *topics* (tags like `python`, `machine-learning`), a good README, and ideally a license.
* **Quality over quantity:** archive or make private any half-finished tutorial copies and test repos. A few complete, documented projects are worth more than 40 empty ones.
* **Meaningful history:** good commit messages, PRs and issues show *how* you work, and that matters as much as the code itself.
* **Never** publish secrets, personal data, or **copyrighted material** (PDF books, course materials).
* **License:** without a `LICENSE` file, nobody can legally reuse your code. GitHub suggests one when you create the repository (MIT is a common, simple choice). [choosealicense.com](https://choosealicense.com/) helps you pick.

#### **Leaving the Playground**

That is the full cycle. The next cell takes the notebook back to its original folder. The playground stays in your temp directory: explore it from a terminal if you like (`cd` into it and try `git log --oneline --graph --all`), or re-run the setup cell to start over.

In [ ]:
%cd {NOTEBOOK_DIR}

# Optional: delete the playground completely
# shutil.rmtree(PLAYGROUND, onexc=_force_remove)

### 7.6 Summary

#### **Cheatsheet**

| Goal | Command |
|---|---|
| **Setup** (once) | `git config --global user.name "..."` / `user.email "..."` |
| Create / copy a repository | `git init` / `git clone <url>` |
| Status now | `git status` (`-s` compact) |
| Stage changes | `git add <file>` / `git add .` / `git add -p` |
| Commit | `git commit -m "Imperative, specific subject"` |
| History | `git log --oneline --graph --all` / `git show <commit>` |
| Changes | `git diff` (unstaged) / `git diff --staged` (staged) |
| Rename / delete / untrack | `git mv` / `git rm` / `git rm --cached` |
| Discard edits in a file | `git restore <file>` ⚠️ |
| Unstage | `git restore --staged <file>` |
| Fix the last (unpushed) commit | `git commit --amend` |
| Undo a pushed commit | `git revert <commit>` |
| Move the branch back | `git reset --soft / --mixed / --hard <commit>` ⚠️ |
| Recover "lost" commits | `git reflog` |
| Put work aside | `git stash` / `git stash pop` |
| Branches | `git branch` / `git switch -c <name>` / `git switch <name>` / `git branch -d <name>` |
| Merge | `git merge <branch>` (conflict: edit → `git add` → `git commit`, or `git merge --abort`) |
| Remotes | `git remote add origin <url>` / `git remote -v` |
| Sync | `git push` (`-u origin <branch>` the first time) / `git fetch` / `git pull` |

#### **Typical Cycle**

```bash
git switch main && git pull               # 1. Start from the latest version
git switch -c feature/thing               # 2. A branch for the task

# edit → git status → git diff → git add → git diff --staged → git commit   (repeat)

git push -u origin feature/thing          # 3. Upload

# 4. Pull Request on GitHub → review → merge

git switch main && git pull               # 5. Update and clean
git branch -d feature/thing
```

#### **Resources**

* [***Pro Git***](https://git-scm.com/book/en/v2): Git's free official book. Chapters 1–3 cover everything here. Chapter 2.4, *"Undoing Things"*, and Chapter 3, *"Git Branching"*, are especially recommended.
* [GitHub Git cheat sheet](https://education.github.com/git-cheat-sheet-education.pdf).
* [Learn Git Branching](https://learngitbranching.js.org/): interactive visual exercises for branches, merge and rebase.
* [Oh Shit, Git!?!](https://ohshitgit.com/): solutions for the typical "I messed up" situations.
* [GitHub Docs: Get started](https://docs.github.com/en/get-started), [About pull requests](https://docs.github.com/en/pull-requests/collaborating-with-pull-requests/proposing-changes-to-your-work-with-pull-requests/about-pull-requests) 
* [Basic writing and formatting syntax (Markdown)](https://docs.github.com/en/get-started/writing-on-github/getting-started-with-writing-and-formatting-on-github/basic-writing-and-formatting-syntax).
* [github/gitignore](https://github.com/github/gitignore): `.gitignore` templates.